# NAM 基线模型对比 - 完整工作流

**仓库**: https://github.com/yaoyuanArtemis/HKU-NAM-7404

本 Notebook 包含完整的实验流程：
1. 从 GitHub 克隆/更新代码
2. 下载 NAM 论文数据集
3. 运行基线模型对比实验（Logistic, CART, XGBoost, DNN-MLP, EBM）
4. **训练 NAM 模型**（可选）
5. 查看和可视化结果
6. 使用 TensorBoard 查看 NAM 训练过程

## 🎯 推荐使用流程

- **快速验证**：运行方案 B（只基线，10-20分钟）
- **完整对比**：运行方案 C（基线+NAM，30-60分钟）⭐ 推荐
- **补充 NAM**：运行方案 D（只NAM，20-40分钟）

## 步骤 1: 克隆项目（首次运行）

**如果已经克隆过项目，跳过此步骤，直接运行「步骤 1.1」更新代码**

In [1]:
import os

# GitHub 配置
GITHUB_REPO = 'https://github.com/yaoyuanArtemis/HKU-NAM-7404.git'
REPO_DIR = '/content/HKU-NAM-7404'
PROJECT_DIR = '/content/HKU-NAM-7404/neural_additive_models'

# 克隆项目
if not os.path.exists(REPO_DIR):
    print("正在从 GitHub 克隆项目...")
    !git clone {GITHUB_REPO} {REPO_DIR}
    print(f"✓ 项目已克隆到: {REPO_DIR}")
else:
    print(f"✓ 项目已存在: {REPO_DIR}")

# 切换到项目目录
os.chdir(PROJECT_DIR)
print(f"\n当前目录: {os.getcwd()}")
print("\n项目文件:")
!ls -lh *.py | head -10

正在从 GitHub 克隆项目...
fatal: could not create leading directories of '/content/HKU-NAM-7404': Read-only file system
✓ 项目已克隆到: /content/HKU-NAM-7404


FileNotFoundError: [Errno 2] No such file or directory: '/content/HKU-NAM-7404/neural_additive_models'

## 步骤 1.1: 更新代码（每次运行前执行）⭐

**重要**: 本地修改代码并 push 到 GitHub 后，在 Colab 中运行此单元格同步最新代码

In [ ]:
import os

# 确保在项目目录
PROJECT_DIR = '/content/HKU-NAM-7404/neural_additive_models'
os.chdir(PROJECT_DIR)

print("🔄 从 GitHub 更新代码...")
!git pull origin master

print(f"\n✓ 代码已更新")
print(f"当前目录: {os.getcwd()}")

# 查看最新提交
print("\n最新提交:")
!git log -1 --oneline

## 步骤 2: 安装依赖

In [ ]:
!pip install -q xgboost scikit-learn interpret pandas numpy matplotlib
print("✓ 依赖安装完成")

## 步骤 3: 下载 NAM 论文数据集 ⭐

**重要**: 运行此步骤下载数据集（自动从公开来源）

In [ ]:
# 下载数据集
!python download_datasets.py

### 查看下载的数据集

In [ ]:
# 列出已下载的数据集
!ls -lh datasets/

## 步骤 4: 运行实验

### 方案 A: 单数据集快速测试

### 方案 B: 批量运行所有数据集 ⭐ 推荐

In [ ]:
# 只运行基线模型（默认）
# 用时：10-20 分钟
!python main.py

### 方案 C: 基线模型 + NAM（完整对比）⭐⭐ 最推荐

**注意**：此方案会训练所有基线模型 + NAM，用时约 30-60 分钟（Colab GPU）

In [ ]:
# 运行基线模型 + NAM
# 用时：30-60 分钟（Colab GPU）
!python main.py --train_nam

### 方案 D: 只运行 NAM（跳过基线）

适用于：已运行过基线模型，只需要补充 NAM 训练

In [ ]:
# 只训练 NAM 模型
# 用时：20-40 分钟（Colab GPU）
!python main.py --only_nam

## 步骤 5: 查看结果

### 查看单个数据集结果

In [ ]:
import pandas as pd
import glob

# 查找结果文件
result_files = glob.glob('comparison_results/*_comparison.csv')

if result_files:
    latest_result = sorted(result_files)[-1]
    print(f"读取结果: {latest_result}\n")
    
    results_df = pd.read_csv(latest_result)
    print("="*70)
    print("模型对比结果")
    print("="*70)
    print(results_df.to_string(index=False))
else:
    print("未找到结果文件，请先运行实验")

### 查看所有数据集汇总

In [ ]:
# 查看批量实验汇总结果
summary_file = 'all_results/ALL_DATASETS_SUMMARY.csv'

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    print("="*70)
    print("所有数据集汇总结果")
    print("="*70)
    print(summary_df.to_string(index=False))
else:
    print("未找到汇总结果，请先运行: python main.py")

## 步骤 6: 可视化结果

In [ ]:
import matplotlib.pyplot as plt

if result_files:
    results_df = pd.read_csv(latest_result)
    
    # 性能对比
    test_metric_cols = [col for col in results_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if test_metric_cols:
        metric_col = test_metric_cols[0]
        
        plt.figure(figsize=(10, 6))
        plt.barh(results_df['Model'], results_df[metric_col])
        plt.xlabel(metric_col)
        plt.title('模型性能对比')
        plt.tight_layout()
        plt.show()
        
        # 训练时间对比
        if 'Train Time (s)' in results_df.columns:
            plt.figure(figsize=(10, 6))
            plt.barh(results_df['Model'], results_df['Train Time (s)'])
            plt.xlabel('训练时间 (秒)')
            plt.title('模型训练时间对比')
            plt.tight_layout()
            plt.show()

### 按数据集可视化（批量实验）

In [ ]:
if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    
    # 找到性能指标列
    metric_cols = [col for col in summary_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if metric_cols:
        metric_col = metric_cols[0]
        
        # 每个数据集的最佳模型
        plt.figure(figsize=(12, 6))
        
        datasets = summary_df['Dataset'].unique()
        for dataset in datasets:
            dataset_df = summary_df[summary_df['Dataset'] == dataset]
            plt.plot(dataset_df['Model'], dataset_df[metric_col], marker='o', label=dataset)
        
        plt.xlabel('Model')
        plt.ylabel(metric_col)
        plt.title('各数据集上的模型性能对比')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 步骤 7: 下载结果

In [ ]:
from google.colab import files

# 打包所有结果
!zip -r results.zip comparison_results/ all_results/ 2>/dev/null || echo "部分目录不存在"

# 下载
files.download('results.zip')

print("✓ 结果已打包并开始下载")

---

## 📚 NAM 论文数据集说明

### ✅ 可自动下载（6个）

1. **Breast Cancer** - sklearn 内置
2. **California Housing** - sklearn 内置
3. **Adult Income** - UCI ML Repository
4. **Heart Disease** - UCI ML Repository
5. **Telco Churn** - IBM 公开数据
6. **COMPAS Recidivism** - ProPublica 公开数据

### ⚠️ 需要手动下载（2个）

7. **Credit Card Fraud**
   - 链接: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
   - 下载后放到 `datasets/creditcard.csv`

8. **FICO Score**
   - 链接: https://community.fico.com/s/explainable-machine-learning-challenge
   - 需要注册

### ❌ 需要授权（1个）

9. **MIMIC-II**
   - 链接: https://mimic.mit.edu/
   - 需要 PhysioNet 认证

---

## 🔄 更新代码工作流

### 本地修改代码后

```bash
cd /Users/sh01679ml/Desktop/workding-code/google-research/neural_additive_models
git add .
git commit -m "更新"
git push origin master
```

### 在 Colab 中更新

重新运行 **步骤 1.1** 即可（自动执行 `git pull`）

## 步骤 8: 查看 NAM 训练结果（可选）

如果运行了 NAM 训练，可以查看训练日志和 TensorBoard

In [ ]:
# 查看 NAM 训练日志目录
!ls -lh all_results/nam_logs/

# 查看某个数据集的训练日志（以 adult 为例）
print("\n=== Adult NAM 训练日志（最后 50 行）===")
!tail -50 all_results/nam_logs/adult/training.log 2>/dev/null || echo "未找到日志文件"

### 使用 TensorBoard 查看 NAM 训练过程

NAM 使用 TensorBoard 记录训练过程，可以可视化查看训练曲线、验证集性能等

---

## 📚 实验说明

### 🎯 三种运行模式

1. **只基线模型**（方案 B）
   - 命令：`python main.py`
   - 用时：10-20 分钟
   - 运行：Logistic, CART, XGBoost, DNN-MLP, EBM

2. **基线 + NAM**（方案 C）⭐ 推荐
   - 命令：`python main.py --train_nam`
   - 用时：30-60 分钟（Colab GPU）
   - 运行：基线模型 + NAM
   - 推荐用于完整论文对比

3. **只 NAM**（方案 D）
   - 命令：`python main.py --only_nam`
   - 用时：20-40 分钟（Colab GPU）
   - 运行：仅 NAM 模型
   - 适用于补充 NAM 训练

---

## 📊 NAM 论文数据集说明

### ✅ 可自动下载（6个）

1. **Breast Cancer** - sklearn 内置
2. **California Housing** - sklearn 内置
3. **Adult Income** - UCI ML Repository
4. **Heart Disease** - UCI ML Repository
5. **Telco Churn** - IBM 公开数据
6. **COMPAS Recidivism** - ProPublica 公开数据

### ⚠️ 需要手动下载（2个）

7. **Credit Card Fraud**
   - 链接: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
   - 下载后放到 `datasets/creditcard.csv`

8. **FICO Score**
   - 链接: https://community.fico.com/s/explainable-machine-learning-challenge
   - 需要注册

### ❌ 需要授权（1个）

9. **MIMIC-II**
   - 链接: https://mimic.mit.edu/
   - 需要 PhysioNet 认证

---

## 🔄 更新代码工作流

### 本地修改代码后

```bash
cd /Users/sh01679ml/Desktop/workding-code/google-research/neural_additive_models
git add .
git commit -m "更新"
git push origin master
```

### 在 Colab 中更新

重新运行 **步骤 1.1** 即可（自动执行 `git pull`）

---

## 💡 使用提示

1. **首次运行**：建议先运行方案 B（只基线），验证流程无误（10分钟）
2. **完整对比**：确认无误后，运行方案 C（基线+NAM），获得完整结果（1小时）
3. **查看结果**：使用步骤 5-6 查看基线结果，步骤 8 查看 NAM 结果
4. **TensorBoard**：NAM 训练可视化在 TensorBoard 中查看
5. **论文对比**：根据 NAM 论文，NAM 应该在大多数数据集上达到接近或超过基线的性能

## 🚀 快捷方式：一键运行全流程

---

# 📚 完整参考指南

以下内容包含所有重要的使用说明、数据集详情、故障排除等信息

## 📊 数据集完整说明

### NAM 论文使用的 9 个数据集

| # | 数据集 | 任务 | 样本数 | 特征数 | 获取方式 |
|---|--------|------|--------|--------|----------|
| 1 | **Breast Cancer** | 分类 | 569 | 30 | sklearn 自动 ✅ |
| 2 | **California Housing** | 回归 | ~20K | 8 | sklearn 自动 ✅ |
| 3 | **Adult Income** | 分类 | ~48K | 14 | UCI 自动 ✅ |
| 4 | **Heart Disease** | 分类 | 303 | 13 | UCI 自动 ✅ |
| 5 | **Telco Churn** | 分类 | ~7K | 19 | IBM 自动 ✅ |
| 6 | **COMPAS Recidivism** | 分类 | ~6K | 11 | ProPublica 自动 ✅ |
| 7 | **Credit Card Fraud** | 分类 | ~284K | 30 | Kaggle 手动 ⚠️ |
| 8 | **FICO Score** | 分类 | ~10K | 24 | FICO 手动 ⚠️ |
| 9 | **MIMIC-II** | 分类 | - | - | 需授权 ❌ |

### 手动下载说明

#### Credit Card Fraud
1. 访问: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
2. 登录 Kaggle 账号
3. 点击 "Download" 下载 `creditcard.csv` (~150MB)
4. 上传到 Colab 或放到本地 `datasets/` 目录

#### FICO Score (HELOC)
1. 访问: https://community.fico.com/s/explainable-machine-learning-challenge
2. 注册账号
3. 下载 `heloc_dataset_v1.csv`
4. 重命名为 `heloc_dataset.csv`
5. 上传到 Colab 或放到本地 `datasets/` 目录

#### MIMIC-II (可选)
- 需要 PhysioNet 认证和数据使用协议 (DUA)
- 访问: https://mimic.mit.edu/
- 建议：可跳过此数据集，其他 8 个数据集已足够

### 数据集详细信息

**Breast Cancer Wisconsin**
- 任务：乳腺癌良恶性分类
- 特征：肿瘤细胞核测量值（半径、纹理、周长等）
- 标签：0=良性，1=恶性

**California Housing**
- 任务：预测加州房价中位数
- 特征：收入、房龄、房间数、地理位置等
- 目标：房价（连续值，单位：$100,000）

**Adult Income (Census)**
- 任务：预测年收入是否超过 $50K
- 特征：年龄、教育、职业、工作时长等
- 标签：<=50K, >50K
- 注意：包含分类变量，需要编码

**Heart Disease**
- 任务：心脏病诊断
- 特征：年龄、血压、胆固醇、心率等
- 标签：0-4（多分类，0=无心脏病）
- 注意：原始数据为多分类，通常转为二分类（0 vs >0）

**Telco Customer Churn**
- 任务：预测客户是否会取消服务
- 特征：服务类型、合同期限、月费用等
- 标签：Yes, No
- 注意：包含分类变量

**COMPAS Recidivism**
- 任务：预测再犯罪风险
- 特征：犯罪历史、年龄、种族等
- 标签：0=未再犯，1=再犯
- 注意：涉及算法公平性问题，敏感数据集

**Credit Card Fraud**
- 任务：信用卡欺诈检测
- 特征：交易金额、时间、PCA 变换后的特征
- 标签：0=正常，1=欺诈
- 注意：高度不平衡（欺诈占 0.172%）

**FICO Score (HELOC)**
- 任务：预测信用评分/违约风险
- 特征：信贷历史、负债比率等
- 标签：Good, Bad
- 注意：金融行业标准数据集

In [ ]:
# 一键：更新代码 + 下载数据 + 运行实验
import os

print("1️⃣ 更新代码...")
os.chdir('/content/HKU-NAM-7404/neural_additive_models')
!git pull origin master

print("\n2️⃣ 下载数据集...")
!python download_datasets.py

print("\n3️⃣ 运行快速测试...")
!python baseline/run_experiment.py \
    --data_path datasets/breast_cancer.csv \
    --target_column target \
    --task classification

print("\n✅ 完成！")

---

## ✅ 完整参考指南总结

本文档是 **NAM 基线模型对比项目** 的完整使用指南，涵盖了从环境配置到结果解读的全流程。

### 📋 快速导航

| 章节 | 内容 | 适用场景 |
|------|------|---------|
| **步骤 1-2** | 环境准备和代码更新 | 首次运行或代码更新 |
| **步骤 3** | 数据集下载 | 获取 NAM 论文数据 |
| **步骤 4** | 运行实验（3种方案） | 核心实验流程 |
| **步骤 5-6** | 查看和可视化结果 | 分析基线模型性能 |
| **步骤 7** | 下载结果 | 保存实验输出 |
| **步骤 8** | NAM 训练结果查看 | 分析 NAM 模型性能 |
| **数据集说明** | 9个数据集详情 | 了解数据特征 |
| **故障排除** | 常见问题解决 | 遇到错误时参考 |
| **性能基准** | NAM 论文预期结果 | 对比实验结果 |
| **命令参考** | 所有可用命令 | 查询具体用法 |
| **架构说明** | 系统设计原理 | 理解代码结构 |
| **结果解读** | 指标和模型对比 | 撰写分析报告 |

### 🎯 核心功能

1. **自动化数据下载**: 6个公开数据集一键下载
2. **基线模型对比**: 5个经典模型（Logistic, CART, XGBoost, DNN, EBM）
3. **NAM 集成训练**: 支持批量 NAM 训练
4. **三种执行模式**: 灵活选择运行内容
5. **完整结果输出**: CSV + Markdown + TensorBoard
6. **可解释性分析**: 支持特征形状函数可视化

### 🚀 推荐工作流

#### 新手入门
```
1. 运行步骤 1-3（环境 + 数据）
2. 运行方案 B（只基线，快速验证）→ 10分钟
3. 查看步骤 5-6（理解结果）
4. 如需 NAM，运行方案 D（补充 NAM）→ 30分钟
```

#### 论文复现
```
1. 确保所有数据集已下载（包括手动的）
2. 运行方案 C（基线 + NAM）→ 1小时
3. 对比 NAM 论文性能基准
4. 生成可视化图表和报告
```

#### 快速测试
```
1. 单数据集测试（run_experiment.py）→ 2分钟
2. 验证流程无误
3. 再运行批量实验
```

### 📊 实验结果概览

当前已运行 **6/9 数据集** 的基线模型对比：

✅ **成功运行**:
- Breast Cancer (分类)
- Adult Income (分类)
- California Housing (回归)
- Credit Card Fraud (分类)
- FICO/HELOC (分类)
- Telco Churn (分类)

❌ **失败数据集** (需修复):
- Heart Disease (多分类问题)
- COMPAS Recidivism (缺失值问题)
- MIMIC-II (未下载)

**最佳模型**: EBM 在 5/6 数据集上表现最佳

**待补充**: NAM 训练结果（预期与 EBM 持平或略优）

### 🔗 相关资源

- **NAM 原论文**: [Neural Additive Models (NeurIPS 2021)](https://arxiv.org/abs/2004.13912)
- **原始代码**: [google-research/neural_additive_models](https://github.com/google-research/neural_additive_models)
- **本项目仓库**: [yaoyuanArtemis/HKU-NAM-7404](https://github.com/yaoyuanArtemis/HKU-NAM-7404)
- **EBM (InterpretML)**: [https://interpret.ml/](https://interpret.ml/)
- **XGBoost 文档**: [https://xgboost.readthedocs.io/](https://xgboost.readthedocs.io/)

### 💡 提示和最佳实践

1. **首次运行**: 建议先运行单个数据集测试，确保环境配置正确
2. **GPU 加速**: NAM 训练强烈建议使用 GPU（Colab 免费 GPU 即可）
3. **结果保存**: 定期下载结果到本地，避免 Colab 会话超时丢失
4. **代码更新**: 每次运行前执行步骤 1.1 同步最新代码
5. **TensorBoard**: NAM 训练时实时查看 TensorBoard 监控进度
6. **内存管理**: 大数据集（Credit Card）训练时注意内存使用

### 🆘 需要帮助？

- **环境问题**: 查看「故障排除」章节
- **数据集问题**: 查看「数据集完整说明」章节
- **性能问题**: 查看「NAM 论文预期性能基准」章节
- **命令用法**: 查看「完整命令参考」章节
- **代码理解**: 查看「系统架构说明」章节
- **结果分析**: 查看「结果解读指南」章节

### 🎓 学习路径

**初级**: 理解 NAM 原理 → 运行单数据集 → 查看结果

**中级**: 批量运行多数据集 → 对比不同模型 → 理解可解释性

**高级**: 修改模型参数 → 添加新数据集 → 扩展新模型

---

**版本信息**: NAM Complete Workflow v2.0  
**更新日期**: 2026-03-06  
**作者**: HKU NAM Project Team  
**联系**: 见项目 GitHub Issues

**Enjoy your NAM experiments!** 🚀

## 📊 结果解读指南

### 理解评估指标

#### 分类任务

**AUROC (Area Under ROC Curve)**
- 范围：0.5 - 1.0
- 0.5：随机猜测
- 0.7-0.8：可接受
- 0.8-0.9：良好
- 0.9+：优秀
- 1.0：完美分类（可能过拟合）

**如何解读**:
```
Train AUROC = 0.999, Test AUROC = 0.920  → 过拟合
Train AUROC = 0.925, Test AUROC = 0.920  → 良好泛化
Train AUROC = 0.850, Test AUROC = 0.845  → 欠拟合（可能需要更复杂模型）
```

#### 回归任务

**RMSE (Root Mean Squared Error)**
- 值越小越好
- 单位与目标变量相同
- 对异常值敏感

**如何解读**:
- California Housing: RMSE = 0.465 表示预测误差约 $46,500（目标单位为 $100,000）
- 比较不同模型时，RMSE 差异 >5% 通常有实际意义

### 模型对比分析

#### 1. 性能维度

**准确性** (Test AUROC/RMSE)
```
EBM: 0.930 ★★★★★
XGBoost: 0.929 ★★★★★
DNN-MLP: 0.908 ★★★☆☆
CART: 0.884 ★★★☆☆
Logistic: 0.861 ★★☆☆☆
```

**训练效率** (Train Time)
```
Logistic: 0.01s  ★★★★★ 最快
CART: 0.02s      ★★★★★
XGBoost: 0.28s   ★★★★☆
DNN-MLP: 1.06s   ★★★☆☆
EBM: 102.21s     ★☆☆☆☆ 最慢
```

**模型复杂度** (Num Parameters)
```
Logistic: 15      ★★★★★ 最简单
CART: 49          ★★★★☆
EBM: 1400         ★★★☆☆
XGBoost: 3100     ★★☆☆☆
DNN-MLP: 12289    ★☆☆☆☆ 最复杂
```

#### 2. 可解释性维度

**全局可解释性**（理解整体模型行为）
```
Logistic/Linear: ★★★★★ 线性权重
CART: ★★★★★ 决策规则
EBM: ★★★★☆ 可加性形状函数
NAM: ★★★★☆ 神经网络形状函数
XGBoost: ★★☆☆☆ 特征重要性
DNN-MLP: ★☆☆☆☆ 黑盒
```

**局部可解释性**（解释单个预测）
```
CART: ★★★★★ 决策路径
EBM/NAM: ★★★★☆ 特征贡献
Logistic: ★★★☆☆ 线性贡献
XGBoost: ★★☆☆☆ SHAP 值
DNN-MLP: ★☆☆☆☆ 需要额外工具
```

### 常见模式识别

#### Pattern 1: 过拟合

**症状**:
```
Model      Train AUROC    Test AUROC    判断
XGBoost    1.000         0.989         轻微过拟合 ⚠️
CART       0.997         0.958         明显过拟合 ❌
```

**原因**: 模型过于复杂，记住了训练数据的噪声
**解决**: 增加正则化、减少模型复杂度、增加数据量

#### Pattern 2: 欠拟合

**症状**:
```
Model      Train AUROC    Test AUROC    判断
Logistic   0.854         0.846         欠拟合 ⚠️
```

**原因**: 模型过于简单，无法捕获数据中的复杂模式
**解决**: 使用更复杂模型、添加特征交互、特征工程

#### Pattern 3: 理想拟合

**症状**:
```
Model      Train AUROC    Test AUROC    判断
EBM        0.937         0.931         良好泛化 ✅
XGBoost    0.940         0.929         良好泛化 ✅
```

**特征**: Train 和 Test 性能接近，Test 性能高

### NAM 特定分析

#### NAM vs 基线模型对比

**查看内容**:
1. **性能对比**: NAM Test AUROC 是否接近或超过最佳基线？
2. **训练时间**: NAM 训练时间是否可接受？（通常 20-40 分钟）
3. **可解释性**: NAM 形状函数是否合理？（在 TensorBoard 中查看）

**示例分析**:
```
数据集: Adult Income
EBM Test AUROC: 0.931      ← 基线最佳
NAM Test AUROC: 0.928      ← NAM（预期）
差异: -0.003 (-0.3%)       ← 可接受范围

结论: NAM 在保持可解释性的同时，达到接近最佳基线的性能 ✅
```

#### TensorBoard 可视化解读

**Training Curve**
- Loss 应该平滑下降
- 如果震荡：学习率可能过高
- 如果平坦：学习率可能过低或已收敛

**Validation Performance**
- 应该随训练改善
- 如果下降：过拟合，需要早停
- 如果不变：欠拟合，需要更多训练

**Feature Shape Functions**
- 应该平滑且有意义
- 锯齿状：需要更多正则化
- 平坦：特征不重要或被压制

### 实际应用建议

#### 场景 1: 生产部署（优先性能）

**推荐模型**: XGBoost 或 EBM
- 高准确性
- 训练可接受
- XGBoost 预测快

#### 场景 2: 监管行业（优先可解释）

**推荐模型**: EBM 或 NAM
- 高可解释性
- 性能接近黑盒
- 符合监管要求（GDPR、金融监管）

#### 场景 3: 快速原型（优先速度）

**推荐模型**: Logistic/CART
- 训练极快
- 易于理解
- 作为基准线

#### 场景 4: 研究/学术（平衡性能和可解释）

**推荐模型**: NAM
- 论文发表价值
- 可解释 + 高性能
- 可视化效果好

### 报告撰写建议

#### 表格展示
```markdown
| 模型 | Test AUROC | 训练时间 | 可解释性 | 推荐场景 |
|------|-----------|---------|---------|---------|
| EBM | 0.931 | 102s | 高 | 监管行业 |
| XGBoost | 0.929 | 0.3s | 中 | 生产部署 |
| NAM | 0.928* | 1800s | 高 | 研究/学术 |
```
*预期结果

#### 可视化
- 条形图：各模型 Test AUROC 对比
- 散点图：性能 vs 训练时间 权衡
- 热力图：不同数据集上的模型表现

#### 文字分析
1. **最佳模型**: 在 X 数据集上，Y 模型达到最佳性能（AUROC = Z）
2. **可解释性**: 在需要可解释性的场景下，推荐 NAM/EBM
3. **效率权衡**: XGBoost 在保持高性能的同时训练速度快 300 倍
4. **结论**: 根据实际应用场景选择合适模型

## 🏗️ 系统架构说明

### 项目结构

本项目基于 Google Research NAM 原始代码，扩展了基线模型对比和批量实验功能。

#### 核心文件

1. **NAM 原始代码**（Google Research）
   - `nam/`：NAM 模型核心实现
   - `nam/nam_train.py`：NAM 训练脚本
   - `nam/graph_builder.py`：计算图构建
   - `nam/data_utils.py`：数据加载模块

2. **基线模型扩展**（本项目新增）
   - `baseline/baseline_models.py`：5 个基线模型实现
     - Logistic Regression / Linear Regression
     - CART (Decision Tree)
     - XGBoost
     - DNN-MLP (Dense Neural Network)
     - EBM (Explainable Boosting Machine)
   
3. **实验框架**（本项目新增）
   - `baseline/run_experiment.py`：单数据集实验脚本
   - `main.py`：批量实验脚本（支持 NAM 集成）
   - `download_datasets.py`：自动数据下载

4. **工作流 Notebook**
   - `NAM_Complete_Workflow.ipynb`：完整工作流（本文档）

### 数据流

```
                    ┌─────────────────────┐
                    │  download_datasets  │
                    │      (下载数据)      │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    datasets/*.csv   │
                    │    (本地 CSV 文件)   │
                    └──────────┬──────────┘
                               │
          ┌────────────────────┼────────────────────┐
          │                    │                    │
          ▼                    ▼                    ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│baseline/         │ │     main.py      │ │ nam/nam_train.py │
│run_experiment.py │ │  (批量实验调度)   │ │  (NAM 单独训练)  │
│ (单数据集基线)    │ │                  │ │                  │
└────────┬─────────┘ └────────┬─────────┘ └────────┬─────────┘
         │                    │                    │
         │                    ├────────────────────┤
         │                    │                    │
         ▼                    ▼                    ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│  *_comparison.csv│ │ALL_DATASETS_     │ │  nam_logs/       │
│  (单数据集结果)   │ │SUMMARY.csv       │ │  (TensorBoard)   │
└──────────────────┘ └──────────────────┘ └──────────────────┘
```

### 关键设计决策

#### 1. NAM 与基线模型分离

**为什么不直接在 baseline_models.py 中实现 NAM？**

- NAM 使用 TensorFlow 1.x，与基线模型的 scikit-learn/PyTorch 环境不同
- NAM 训练需要特定数据格式和大量超参数
- NAM 原始代码已有完整实现，重用更可靠
- 保持模块化，便于独立调试和维护

**实现方式**: `main.py` 通过 `subprocess` 调用 `nam/nam_train.py`

#### 2. 三种执行模式

```python
# main.py 中的逻辑
def run_experiment_on_dataset(..., train_nam=False, only_nam=False):
    if not only_nam:
        # 运行基线模型 (baseline/run_experiment.py)
        baseline_success = run_baseline_models()
    
    if train_nam or only_nam:
        # 训练 NAM (nam/nam_train.py)
        nam_success = train_nam_on_dataset()
    
    return baseline_success or nam_success
```

**优势**:
- 灵活性：可选择只跑基线、只跑 NAM、或两者都跑
- 容错性：基线失败不影响 NAM，反之亦然
- 效率：已有基线结果时可只补充 NAM

#### 3. 数据集配置统一管理

```python
DATASETS = {
    'BreastCancer': {
        'file': 'breast_cancer.csv',      # 本地文件名
        'task': 'classification',          # 任务类型
        'target': 'target',                # 目标列名
        'nam_dataset_name': 'BreastCancer' # NAM 内置名称
    },
    # ...
}
```

**作用**:
- 单一配置源，避免重复和不一致
- 同时支持本地 CSV（基线）和 NAM 内置格式
- 便于添加新数据集

### 数据处理流程

#### 基线模型数据流

```
CSV 文件 → pandas DataFrame → train/val/test split
         ↓
    LabelEncoder (分类变量)
         ↓
    StandardScaler (数值标准化)
         ↓
    scikit-learn / XGBoost / interpret 模型
         ↓
    评估指标计算 (AUROC/RMSE)
         ↓
    CSV 结果文件
```

#### NAM 数据流

```
CSV 文件 → NAM data_utils 加载器
         ↓
    NAM 特定预处理
         ↓
    TensorFlow 1.x 计算图
         ↓
    训练 (TensorBoard 日志)
         ↓
    模型保存 + 评估结果
```

### 扩展指南

#### 添加新的基线模型

编辑 `baseline/baseline_models.py`:
```python
class BaselineComparison:
    def train_and_evaluate(...):
        # 添加新模型
        models['NewModel'] = YourModelClass()
        # ...
```

#### 添加新的数据集

1. 下载数据到 `datasets/your_data.csv`
2. 编辑 `main.py`:
```python
DATASETS['YourData'] = {
    'file': 'your_data.csv',
    'task': 'classification',
    'target': 'label_column',
    'nam_dataset_name': 'YourData'  # 需在 NAM 中注册
}
```

3. 如果要训练 NAM，需在 `nam/nam_train.py` 中注册数据集加载逻辑

### 已知限制

1. **TensorFlow 版本**: NAM 依赖 TensorFlow 1.x，与 TF 2.x 不兼容
2. **GPU 内存**: 大数据集训练 NAM 需要充足 GPU 内存
3. **数据格式**: NAM 对数据格式有严格要求（需特定预处理）
4. **多分类支持**: 部分数据集（Heart Disease）需要额外处理
5. **缺失值**: 某些数据集（Recidivism）包含 NaN，需要插补

## 📚 完整命令参考

### 1. 数据下载

```bash
python download_datasets.py
```

**功能**: 自动下载 6 个可公开获取的数据集
- Breast Cancer (sklearn)
- California Housing (sklearn)
- Adult Income (UCI)
- Heart Disease (UCI)
- Telco Churn (IBM)
- COMPAS Recidivism (ProPublica)

### 2. 批量实验运行

#### 方案 A: 只运行基线模型（默认）
```bash
python main.py
```
- 用时：10-20 分钟
- 运行模型：Logistic/Linear, CART, XGBoost, DNN-MLP, EBM
- 输出：`all_results/*_comparison.csv`

#### 方案 B: 运行基线模型 + NAM
```bash
python main.py --train_nam
```
- 用时：30-60 分钟（Colab GPU）
- 运行：基线模型 + NAM
- 输出：基线结果 + NAM 日志（`all_results/nam_logs/`）

#### 方案 C: 只运行 NAM（跳过基线）
```bash
python main.py --only_nam
```
- 用时：20-40 分钟（Colab GPU）
- 适用场景：已有基线结果，只需补充 NAM
- 输出：NAM 训练日志

### 3. 单数据集实验

```bash
python baseline/run_experiment.py \
    --data_path datasets/breast_cancer.csv \
    --target_column target \
    --task classification \
    --output_dir ./results
```

**参数说明**:
- `--data_path`: 数据集文件路径（必需）
- `--target_column`: 目标列名称（必需）
- `--task`: 任务类型，`classification` 或 `regression`（必需）
- `--output_dir`: 结果输出目录（可选，默认 `./comparison_results`）

### 4. NAM 单独训练

```bash
python nam/nam_train.py \
    --dataset_name BreastCancer \
    --training_epochs 1000 \
    --learning_rate 0.01 \
    --batch_size 1024 \
    --dropout 0.5 \
    --logdir ./nam_logs/breast_cancer \
    --regression false
```

**参数说明**:
- `--dataset_name`: NAM 内置数据集名称（BreastCancer, Adult, Housing 等）
- `--training_epochs`: 训练轮数（默认 1000）
- `--learning_rate`: 学习率（默认 0.01）
- `--batch_size`: 批大小（默认 1024）
- `--dropout`: Dropout 比率（默认 0.5）
- `--logdir`: TensorBoard 日志目录
- `--regression`: 是否回归任务（`true` 或 `false`）
- `--output_regularization`: 输出正则化（默认 0.0）
- `--l2_regularization`: L2 正则化（默认 0.0）

### 5. 查看 TensorBoard

```bash
tensorboard --logdir all_results/nam_logs/
```

**或在 Jupyter/Colab 中**:
```python
%load_ext tensorboard
%tensorboard --logdir all_results/nam_logs/
```

### 6. 结果文件结构

```
neural_additive_models/
│
├── main.py                            # 主程序
├── download_datasets.py               # 数据下载
├── NAM_Complete_Workflow.ipynb        # 完整工作流
│
├── nam/                               # NAM 核心代码
│   ├── nam_train.py
│   ├── graph_builder.py
│   ├── models.py
│   └── data_utils.py
│
├── baseline/                          # 基线模型
│   ├── baseline_models.py
│   └── run_experiment.py
│
├── datasets/                          # 数据集
│   ├── breast_cancer.csv
│   ├── adult.csv
│   └── ...
│
└── all_results/                       # 实验结果
    ├── ALL_DATASETS_SUMMARY.csv
    ├── SUMMARY_REPORT.md
    ├── *_comparison.csv
    └── nam_logs/
        ├── adult/
        └── breast_cancer/
```

## 📈 NAM 论文预期性能基准

根据 NAM 原论文（Agarwal et al., 2021），NAM 模型在大多数数据集上应该达到接近或超过基线模型的性能，同时保持可解释性。

### 分类任务预期 AUROC

| 数据集 | NAM (论文) | DNN | XGBoost | EBM | 说明 |
|--------|-----------|-----|---------|-----|------|
| **Breast Cancer** | ~0.995 | ~0.990 | ~0.992 | ~0.993 | NAM 与最佳模型接近 |
| **Adult** | ~0.930 | ~0.910 | ~0.928 | ~0.925 | NAM 达到最佳性能 |
| **Heart Disease** | ~0.900 | ~0.880 | ~0.890 | ~0.895 | 小数据集，性能相近 |
| **Telco Churn** | ~0.845 | ~0.830 | ~0.835 | ~0.845 | NAM 与 EBM 并列第一 |
| **COMPAS** | ~0.725 | ~0.710 | ~0.720 | ~0.720 | 敏感数据集，性能适中 |
| **Credit Card** | ~0.975 | ~0.970 | ~0.975 | ~0.975 | 高度不平衡数据集 |
| **FICO** | ~0.795 | ~0.785 | ~0.790 | ~0.795 | NAM 与 EBM 持平 |

### 回归任务预期 RMSE

| 数据集 | NAM (论文) | DNN | XGBoost | EBM | 说明 |
|--------|-----------|-----|---------|-----|------|
| **California Housing** | ~0.465 | ~0.525 | ~0.490 | ~0.465 | NAM 与 EBM 达到最佳 |

### 关键发现

1. **可解释性 + 性能**: NAM 在保持高可解释性的同时，达到接近黑盒模型（DNN、XGBoost）的性能

2. **与 EBM 对比**: NAM 和 EBM 性能接近，但 NAM 使用神经网络，训练更灵活

3. **优势场景**:
   - 大规模数据集（Adult、Credit Card）
   - 需要可解释性的场景（COMPAS、FICO）
   - 回归任务（California Housing）

4. **注意事项**:
   - 小数据集（Heart Disease）性能提升有限
   - 训练时间比简单模型（Logistic、CART）长
   - 需要 GPU 加速效果更好

### 实际运行结果对比

根据本次实验的实际结果（见 `all_results/SUMMARY_REPORT.md`）:

**已运行的基线模型表现** (6个数据集):
- Adult: EBM 最佳 (0.931)
- Breast Cancer: Logistic 最佳 (0.996)
- California Housing: EBM 最佳 (0.466 RMSE)
- Credit Card: EBM 最佳 (0.976)
- FICO: EBM 最佳 (0.804)
- Telco: EBM 最佳 (0.846)

**待补充**: NAM 模型训练结果（使用 `--train_nam` 或 `--only_nam`）

## 🔧 常见问题和故障排除

### 问题 1: XGBoost OpenMP 错误 (macOS)

**错误信息**:
```
XGBoostError: XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes: * OpenMP runtime is not installed
```

**解决方案**:
```bash
brew install libomp
```

### 问题 2: Python 命令未找到

**错误信息**:
```
[Errno 2] No such file or directory: 'python'
```

**原因**: macOS 默认只有 `python3`，没有 `python` 命令

**解决方案**: 已在代码中修复，使用 `sys.executable` 替代硬编码的 `python`

### 问题 3: 字符串标签错误

**错误信息**:
```
ValueError: Invalid classes inferred from unique values of y.
Expected: [0 1], got ['<=50K' '>50K']
```

**原因**: 某些数据集（如 Adult）的标签是字符串

**解决方案**: 已在 `run_experiment.py` 中自动添加 LabelEncoder 处理

### 问题 4: 缺少 tabulate 包

**错误信息**:
```
ImportError: Missing optional dependency 'tabulate'
```

**解决方案**:
```bash
pip install tabulate
```

### 问题 5: Heart Disease 多分类错误

**错误信息**:
```
xgboost.core.XGBoostError: label and prediction size not match
```

**原因**: Heart Disease 数据集是多分类（0-4），但被当作二分类处理

**解决方案**: 需要在数据预处理时转换为二分类（0 vs >0），或使用 `multi:softprob`

### 问题 6: Recidivism 数据包含 NaN

**错误信息**:
```
ValueError: Input X contains NaN.
LogisticRegression does not accept missing values
```

**原因**: COMPAS 数据集包含缺失值

**解决方案**: 需要在预处理时添加插补步骤（imputation）或删除缺失值

### 问题 7: Colab 文件系统只读

**错误信息**:
```
fatal: could not create leading directories of '/content/HKU-NAM-7404': Read-only file system
```

**原因**: 尝试在只读目录创建文件

**解决方案**: 使用 Colab 默认的 `/content` 目录作为工作目录